In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pycountry
import pickle
from datetime import datetime

In [2]:
df = pd.read_csv('../owid_covid_data.csv')

with open('countries.pkl', 'rb') as f:
    countries_kept = pickle.load(f)

df = df[df['country'].isin(countries_kept)]

df = df[df['date'] <= '2022-05-19']

weekday = lambda date: datetime.strptime(date, '%Y-%m-%d').weekday()
df = df[df['date'].apply(weekday) == 0]

df = df.reset_index(drop = True)

columns = ['country', 'new_cases_smoothed', 'population', 'population_density', 'median_age', 'gdp_per_capita', 'hospital_beds_per_thousand', 'life_expectancy']

df = df[columns]

for col in ['population_density', 'gdp_per_capita', 'hospital_beds_per_thousand']:
    df[col] = np.log(df[col])
    df = df.rename(columns={col: 'log_' + col})

In [3]:
#Generated by ChatGPT
country_to_continent = {
    'Afghanistan': 'Asia',
    'Albania': 'Europe',
    'Algeria': 'Africa',
    'Antigua and Barbuda': 'North America',
    'Argentina': 'South America',
    'Armenia': 'Asia',
    'Australia': 'Oceania',
    'Austria': 'Europe',
    'Azerbaijan': 'Asia',
    'Bahamas': 'North America',
    'Bahrain': 'Asia',
    'Bangladesh': 'Asia',
    'Barbados': 'North America',
    'Belarus': 'Europe',
    'Belgium': 'Europe',
    'Belize': 'North America',
    'Bhutan': 'Asia',
    'Bolivia': 'South America',
    'Bosnia and Herzegovina': 'Europe',
    'Brazil': 'South America',
    'Brunei': 'Asia',
    'Bulgaria': 'Europe',
    'Burundi': 'Africa',
    'Cambodia': 'Asia',
    'Canada': 'North America',
    'Central African Republic': 'Africa',
    'Chile': 'South America',
    'China': 'Asia',
    'Colombia': 'South America',
    'Costa Rica': 'North America',
    'Croatia': 'Europe',
    'Cyprus': 'Europe',
    'Czechia': 'Europe',
    'Denmark': 'Europe',
    'Djibouti': 'Africa',
    'Dominica': 'North America',
    'Dominican Republic': 'North America',
    'Ecuador': 'South America',
    'Egypt': 'Africa',
    'El Salvador': 'North America',
    'Estonia': 'Europe',
    'Eswatini': 'Africa',
    'Ethiopia': 'Africa',
    'Fiji': 'Oceania',
    'Finland': 'Europe',
    'France': 'Europe',
    'Gambia': 'Africa',
    'Georgia': 'Asia',
    'Germany': 'Europe',
    'Ghana': 'Africa',
    'Greece': 'Europe',
    'Grenada': 'North America',
    'Guatemala': 'North America',
    'Guinea': 'Africa',
    'Guyana': 'South America',
    'Haiti': 'North America',
    'Honduras': 'North America',
    'Hungary': 'Europe',
    'Iceland': 'Europe',
    'India': 'Asia',
    'Indonesia': 'Asia',
    'Iran': 'Asia',
    'Iraq': 'Asia',
    'Ireland': 'Europe',
    'Israel': 'Asia',
    'Italy': 'Europe',
    'Jamaica': 'North America',
    'Japan': 'Asia',
    'Jordan': 'Asia',
    'Kazakhstan': 'Asia',
    'Kiribati': 'Oceania',
    'Kuwait': 'Asia',
    'Kyrgyzstan': 'Asia',
    'Laos': 'Asia',
    'Latvia': 'Europe',
    'Lebanon': 'Asia',
    'Libya': 'Africa',
    'Lithuania': 'Europe',
    'Luxembourg': 'Europe',
    'Malawi': 'Africa',
    'Malaysia': 'Asia',
    'Malta': 'Europe',
    'Mauritius': 'Africa',
    'Mexico': 'North America',
    'Moldova': 'Europe',
    'Mongolia': 'Asia',
    'Montenegro': 'Europe',
    'Morocco': 'Africa',
    'Mozambique': 'Africa',
    'Myanmar': 'Asia',
    'Nepal': 'Asia',
    'Netherlands': 'Europe',
    'New Zealand': 'Oceania',
    'Nicaragua': 'North America',
    'Niger': 'Africa',
    'North Macedonia': 'Europe',
    'Norway': 'Europe',
    'Oman': 'Asia',
    'Pakistan': 'Asia',
    'Panama': 'North America',
    'Paraguay': 'South America',
    'Peru': 'South America',
    'Philippines': 'Asia',
    'Poland': 'Europe',
    'Portugal': 'Europe',
    'Qatar': 'Asia',
    'Romania': 'Europe',
    'Russia': 'Europe',
    'Saint Kitts and Nevis': 'North America',
    'Saint Lucia': 'North America',
    'Saint Vincent and the Grenadines': 'North America',
    'San Marino': 'Europe',
    'Sao Tome and Principe': 'Africa',
    'Saudi Arabia': 'Asia',
    'Serbia': 'Europe',
    'Seychelles': 'Africa',
    'Singapore': 'Asia',
    'Slovakia': 'Europe',
    'Slovenia': 'Europe',
    'Solomon Islands': 'Oceania',
    'Somalia': 'Africa',
    'South Korea': 'Asia',
    'Spain': 'Europe',
    'Sri Lanka': 'Asia',
    'Sudan': 'Africa',
    'Suriname': 'South America',
    'Sweden': 'Europe',
    'Switzerland': 'Europe',
    'Tajikistan': 'Asia',
    'Togo': 'Africa',
    'Trinidad and Tobago': 'North America',
    'Tunisia': 'Africa',
    'Turkey': 'Asia',
    'Turkmenistan': 'Asia',
    'Ukraine': 'Europe',
    'United Arab Emirates': 'Asia',
    'United Kingdom': 'Europe',
    'United States': 'North America',
    'Uruguay': 'South America',
    'Uzbekistan': 'Asia',
    'Vietnam': 'Asia',
    'Zimbabwe': 'Africa'
}

df['continent'] = df['country'].apply(lambda country: country_to_continent[country])

In [4]:
df = df.fillna(0)

In [5]:
df_countries = df.drop_duplicates(subset = 'country', keep = 'last').copy()
X = np.array(df_countries[['log_population_density', 'median_age', 'log_gdp_per_capita', 'log_hospital_beds_per_thousand', 'life_expectancy']])

In [6]:
#Generated by ChatGPT
X_std = (X - X.mean(axis = 0)) / X.std(axis = 0, ddof = 1)

U, S, Vt = np.linalg.svd(X_std, full_matrices=False)

df_countries['pc'] = X_std @ Vt[0]

In [7]:
df = df.merge(df_countries[['country', 'pc']], on = 'country', how = 'left')
df = df[['country', 'continent', 'new_cases_smoothed', 'pc', 'population']]
df

,country,continent,new_cases_smoothed,pc,population
0,Afghanistan,Asia,0.000000,-3.625447,40578847.0
1,Afghanistan,Asia,0.000000,-3.625447,40578847.0
2,Afghanistan,Asia,0.000000,-3.625447,40578847.0
3,Afghanistan,Asia,0.000000,-3.625447,40578847.0
4,Afghanistan,Asia,0.000000,-3.625447,40578847.0
...,...,...,...,...,...
17603,Zimbabwe,Africa,43.571430,-2.790839,16069061.0
17604,Zimbabwe,Africa,39.428570,-2.790839,16069061.0
17605,Zimbabwe,Africa,41.142857,-2.790839,16069061.0
17606,Zimbabwe,Africa,72.857140,-2.790839,16069061.0
